<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# GRU for Sequential Recommendation

This notebook trains a GRU \[1\] sequential recommender on a public Amazon Movies & TV review dataset. The GRU model encodes a user's history of (item, category) interactions with a Gated Recurrent Unit, then combines the resulting hidden state with the target item / category / user embeddings to predict whether the user will interact with the target next.

The implementation is PyTorch-native; it does not depend on the legacy TF `BaseModel` / `SequentialBaseModel` hierarchy used by the other sequential models in this repository (see [`sequential_recsys_amazondataset.ipynb`](sequential_recsys_amazondataset.ipynb) for those). It mirrors the single-file standalone `nn.Module` convention introduced by the LightGCN PyTorch migration.



## Architecture

**Inputs.** Each instance is a tuple `(user, target_item, target_category, item_history, category_history, mask)`. The histories are padded to a fixed `max_seq_length` `T`; the binary `mask` marks valid time steps.

**Per-step input.** At each time step $t$, the item and category embeddings are concatenated to form the GRU input:

$$
x_t = \big[\,\mathrm{emb}_{\text{item}}(\text{item\_history}_t)\;;\;\mathrm{emb}_{\text{cate}}(\text{cate\_history}_t)\,\big].
$$

**GRU recurrence.** The standard GRU update is applied for $t = 1, \dots, T$:

$$
\begin{aligned}
z_t &= \sigma(W_z x_t + U_z h_{t-1}) & \text{(update gate)} \\
r_t &= \sigma(W_r x_t + U_r h_{t-1}) & \text{(reset gate)} \\
\tilde{h}_t &= \tanh(W_h x_t + U_h (r_t \odot h_{t-1})) & \text{(candidate)} \\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t & \text{(hidden)}
\end{aligned}
$$

Padded steps are skipped by `pack_padded_sequence`, so only valid history positions contribute to the final hidden state $h_T$.

**Prediction head.** The final hidden state is concatenated with three per-instance, time-invariant features &mdash; the target item, the target category, and the user embedding &mdash; and fed through an MLP (`Linear` → `BatchNorm1d` → activation → dropout, repeated `len(layer_sizes)` times, then a final `Linear` to 1):

$$
\text{logit} = \mathrm{FCN}\big(\big[\,h_T\;;\;\mathrm{emb}_{\text{item}}(\text{target\_item})\;;\;\mathrm{emb}_{\text{cate}}(\text{target\_cate})\;;\;\mathrm{emb}_{\text{user}}(\text{user})\,\big]\big).
$$

**Loss.** Each positive instance is grouped with `train_num_ngs` in-batch negatives (the iterator emits the layout `[positive, neg_1, ..., neg_N, positive, neg_1, ...]`). The training loss is the standard softmax over the resulting group:

$$
\mathcal{L} = -\,(1 + N) \cdot \frac{1}{B}\sum_{b=1}^{B} \log \frac{\exp(\text{logit}_b^{+})}{\sum_{j=0}^{N}\exp(\text{logit}_b^{(j)})},
$$

plus L1 / L2 regularization on the embedding tables and the GRU / MLP parameters.


In [19]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import os
import sys

import torch

from recommenders.utils.timer import Timer
from recommenders.utils.constants import SEED
from recommenders.datasets.amazon_reviews import download_and_extract, data_preprocessing
from recommenders.models.deeprec.io.torch.sequential_iterator import SequentialIterator
from recommenders.models.deeprec.models.sequential.gru import GRUModel
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")


System version: 3.11.0 (main, Mar 16 2025, 13:39:19) [GCC 11.4.0]
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1


### Input Parameters

`GRUModel` separates *architectural* hyper-parameters (those that shape the parameter tensors) from *training* hyper-parameters (those passed to `fit()`). No YAML config is required.


In [21]:
EPOCHS = 10
BATCH_SIZE = 400
RANDOM_SEED = SEED  # Set None for non-deterministic result

# Architectural hyper-parameters (passed to GRUModel.__init__)
MAX_SEQ_LENGTH = 50
HIDDEN_SIZE = 40
USER_EMBEDDING_DIM = 16
ITEM_EMBEDDING_DIM = 32
CATE_EMBEDDING_DIM = 8
LAYER_SIZES = [100, 64]
ACTIVATIONS = ["relu", "relu"]
DROPOUT = [0.3, 0.3]
ENABLE_BN = True

# Training hyper-parameters (passed to GRUModel.fit)
LEARNING_RATE = 0.001
EMBED_L2 = 0.0
LAYER_L2 = 0.0
TRAIN_NUM_NGS = 4   # in-batch negatives per positive during training
VALID_NUM_NGS = 4   # negatives per positive in valid_data
TEST_NUM_NGS = 9    # negatives per positive in test_data

data_path = os.path.join("..", "..", "tests", "resources", "deeprec", "slirec")


## 1. Input data format

The input data has 8 tab-separated columns:

```
<label>\t<user_id>\t<item_id>\t<category_id>\t<timestamp>\t<history_item_ids>\t<history_category_ids>\t<history_timestamps>
```

History columns are comma-separated lists. One example row:

```
1   A1QQ86H5M2LVW2   B0059XTU1S   Movies   1377561600   B002ZG97WE,B004IK30PA,...   Movies,Movies,...   1304294400,1304812800,...
```

In data preprocessing, IDs are mapped to integer indices and saved as vocab pickles (`user_vocab.pkl`, `item_vocab.pkl`, `category_vocab.pkl`). New IDs in valid/test files that were not seen during training are mapped to 0.

For training and evaluation, each positive instance is followed by `num_ngs` negative instances; the model consumes them as a `(1 + num_ngs)` group for softmax loss. If you only have positive instances, pass `need_sample=True` semantics by setting `train_num_ngs=4` &mdash; the iterator samples in-batch negatives at training time.

### Amazon Movies & TV dataset

The cell below downloads a small sample of the public Amazon Movies & TV review dataset and runs the standard preprocessing pipeline.


In [22]:
train_file = os.path.join(data_path, "train_data")
valid_file = os.path.join(data_path, "valid_data")
test_file = os.path.join(data_path, "test_data")
user_vocab = os.path.join(data_path, "user_vocab.pkl")
item_vocab = os.path.join(data_path, "item_vocab.pkl")
cate_vocab = os.path.join(data_path, "category_vocab.pkl")
output_file = os.path.join(data_path, "output.txt")

reviews_name = "reviews_Movies_and_TV_5.json"
meta_name = "meta_Movies_and_TV.json"
reviews_file = os.path.join(data_path, reviews_name)
meta_file = os.path.join(data_path, meta_name)
sample_rate = 0.01  # subsample for a quick example

input_files = [reviews_file, meta_file, train_file, valid_file, test_file,
               user_vocab, item_vocab, cate_vocab]

if not os.path.exists(train_file):
    download_and_extract(reviews_name, reviews_file)
    download_and_extract(meta_name, meta_file)
    data_preprocessing(
        *input_files,
        sample_rate=sample_rate,
        valid_num_ngs=VALID_NUM_NGS,
        test_num_ngs=TEST_NUM_NGS,
    )


## 2. Data loader

`SequentialIterator` parses the tab-separated file format described above and yields batches as `dict[str, np.ndarray]`. The iterator is TF-free; the GRU model converts the numpy arrays to PyTorch tensors internally.


In [23]:
input_creator = SequentialIterator(
    user_vocab=user_vocab,
    item_vocab=item_vocab,
    cate_vocab=cate_vocab,
    max_seq_length=MAX_SEQ_LENGTH,
    batch_size=BATCH_SIZE,
)
print(f"#users={input_creator.user_vocab_length}, "
      f"#items={input_creator.item_vocab_length}, "
      f"#cates={input_creator.cate_vocab_length}")


#users=3487, #items=475, #cates=14


## 3. Create the model

Architectural hyper-parameters are passed to the constructor; training-time hyper-parameters live on `fit()`.


In [24]:
model = GRUModel(
    user_vocab_length=input_creator.user_vocab_length,
    item_vocab_length=input_creator.item_vocab_length,
    cate_vocab_length=input_creator.cate_vocab_length,
    user_embedding_dim=USER_EMBEDDING_DIM,
    item_embedding_dim=ITEM_EMBEDDING_DIM,
    cate_embedding_dim=CATE_EMBEDDING_DIM,
    max_seq_length=MAX_SEQ_LENGTH,
    hidden_size=HIDDEN_SIZE,
    layer_sizes=LAYER_SIZES,
    activations=ACTIVATIONS,
    dropout=DROPOUT,
    enable_BN=ENABLE_BN,
    seed=RANDOM_SEED,
)

# Attach the iterator + group size so we can call run_eval *before* fit().
# fit() also installs an iterator built from the vocab paths; this attachment
# only matters for the pre-training evaluation cell below.
model.iterator = input_creator
model.train_num_ngs = TRAIN_NUM_NGS


### 3.1 Train

`fit()` runs `EPOCHS` epochs of training on `train_file`, evaluating on `valid_file` each epoch and tracking `eval_metric` (default `group_auc`) for early stopping.


In [25]:
with Timer() as train_time:
    model = model.fit(
        train_file=train_file,
        valid_file=valid_file,
        user_vocab=user_vocab,
        item_vocab=item_vocab,
        cate_vocab=cate_vocab,
        valid_num_ngs=VALID_NUM_NGS,
        train_num_ngs=TRAIN_NUM_NGS,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        embed_l2=EMBED_L2,
        layer_l2=LAYER_L2,
        show_step=20,
        save_model=True,
        save_epoch=1,
        model_dir=os.path.join(data_path, "model"),
    )

print(f"Time cost for training is {train_time.interval / 60.0:.2f} mins")


Time cost for training is 1.86 mins


### 3.2 Evaluate

After training, AUC should be well above 0.5 and the group-wise metrics (`mean_mrr`, `ndcg@k`, `group_auc`) should reflect the model's ranking quality on the held-out test set.


In [26]:
res_syn = model.run_eval(test_file, num_ngs=TEST_NUM_NGS)
print(res_syn)

{'auc': 0.6256, 'logloss': 0.7521, 'mean_mrr': 0.3928, 'ndcg@10': 0.5346, 'group_auc': 0.6248}


In [27]:
# Record results for tests - ignore this cell
store_metadata("auc", res_syn["auc"])
store_metadata("logloss", res_syn["logloss"])
store_metadata("mean_mrr", res_syn["mean_mrr"])
store_metadata("ndcg@10", res_syn["ndcg@10"])
store_metadata("group_auc", res_syn["group_auc"])


### 3.3 Predict

`predict()` writes one score per test row to `output_file`.


In [28]:
model.predict(test_file, output_file)

GRUModel(
  (user_embedding): Embedding(3487, 16)
  (item_embedding): Embedding(475, 32)
  (cate_embedding): Embedding(14, 8)
  (gru): GRU(40, 40, batch_first=True)
  (fcn): _FCN(
    (linears): ModuleList(
      (0): Linear(in_features=96, out_features=100, bias=True)
      (1): Linear(in_features=100, out_features=64, bias=True)
    )
    (bns): ModuleList(
      (0): BatchNorm1d(100, eps=0.0001, momentum=0.05, affine=True, track_running_stats=True)
      (1): BatchNorm1d(64, eps=0.0001, momentum=0.05, affine=True, track_running_stats=True)
    )
    (out): Linear(in_features=64, out_features=1, bias=True)
  )
)

## 4. Loading a saved checkpoint

`fit(save_model=True)` writes `best_model.pt` (the checkpoint of the best epoch by `eval_metric`) and `epoch_{N}_model.pt` (per-epoch checkpoints) under `model_dir`. To restore the best checkpoint:


In [29]:
model_best_trained = GRUModel(
    user_vocab_length=input_creator.user_vocab_length,
    item_vocab_length=input_creator.item_vocab_length,
    cate_vocab_length=input_creator.cate_vocab_length,
    user_embedding_dim=USER_EMBEDDING_DIM,
    item_embedding_dim=ITEM_EMBEDDING_DIM,
    cate_embedding_dim=CATE_EMBEDDING_DIM,
    max_seq_length=MAX_SEQ_LENGTH,
    hidden_size=HIDDEN_SIZE,
    layer_sizes=LAYER_SIZES,
    activations=ACTIVATIONS,
    dropout=DROPOUT,
    enable_BN=ENABLE_BN,
    seed=RANDOM_SEED,
)

# Pass a directory: load() picks up best_model.pt by default.
# Pass filename=f"epoch_3_{MODEL_CHECKPOINT}" (from the gru module) to load a specific epoch.
model_dir = os.path.join(data_path, "model")
print(f"loading saved model in {model_dir}")
model_best_trained.load(model_dir)

# Re-attach the iterator so run_eval / predict have data to consume.
model_best_trained.iterator = input_creator
model_best_trained.train_num_ngs = TRAIN_NUM_NGS

print(model_best_trained.run_eval(test_file, num_ngs=TEST_NUM_NGS))


loading saved model in ../../tests/resources/deeprec/slirec/model
{'auc': 0.6371, 'logloss': 0.8046, 'mean_mrr': 0.3958, 'ndcg@10': 0.5372, 'group_auc': 0.6318}


## References

\[1\] Kyunghyun Cho, Bart van Merrienboer, Caglar Gulcehre, Dzmitry Bahdanau, Fethi Bougares, Holger Schwenk, and Yoshua Bengio. *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. arXiv preprint arXiv:1406.1078. 2014.

\[2\] [Amazon Movies & TV reviews dataset](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Movies_and_TV_5.json.gz) and [metadata](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_Movies_and_TV.json.gz).
